# Task 1: Source Discovery and Data Preparation

In this task, the assigned chapter is used as the knowledge base for the Retrieval-Augmented Generation (RAG) system. According to the assignment instructions, the chapter is determined by the last digit of the student ID. Since my student ID ends with 7, the assigned chapter is Chapter 7. The PDF file for this chapter has already been downloaded and stored locally as `assets/Chapter_7.pdf`.

The main objective of this task is to prepare the chapter content so that it can later be used in both Naive RAG and Contextual Retrieval. To achieve this, the text must first be extracted from the PDF, cleaned to remove unwanted formatting noise, and then divided into manageable chunks for retrieval. In addition, at least 20 question-answer pairs must be created based strictly on the content of the assigned chapter. These QA pairs will later serve as evaluation queries for comparing the two retrieval methods.

## Importing Required Libraries

This section imports the libraries needed for Task 1. The `Path` class is used to define file paths in a clean and platform-independent way. The `re` module is used for text cleaning with regular expressions, while `json` will later help save processed outputs such as chunks and question-answer pairs. The `fitz` library from PyMuPDF is used to open the chapter PDF and extract text page by page. Finally, `pprint` is imported as a helper for previewing structured outputs during development.

In [1]:
# standard library imports
from pathlib import Path
import re
import json

# PDF reading library
import fitz  # PyMuPDF

# Optional display helpers
from pprint import pprint

## Defining Input and Output Paths

This section defines the location of the source PDF and the output files that will be created during preprocessing. The chapter PDF is stored in the `assets` folder, while all generated outputs are saved into an `artefacts` folder. This makes the workflow more organized and reproducible. Separate files are prepared for the raw extracted text, cleaned text, chunked text, and the final QA pairs.

In [2]:
# Define the project paths
PROJECT_ROOT = Path(".")
ASSETS_DIR = PROJECT_ROOT / "assets"
OUTPUT_DIR = PROJECT_ROOT / "artefacts"

# Create output directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define the input PDF path
PDF_PATH = ASSETS_DIR / "Chapter_7.pdf"

# Define output file paths
RAW_TEXT_PATH = OUTPUT_DIR / "chapter_7_raw.txt"
CLEAN_TEXT_PATH = OUTPUT_DIR / "chapter_7_cleaned.txt"
CHUNKS_PATH = OUTPUT_DIR / "chapter_7_chunks.json"
QA_PATH = OUTPUT_DIR / "qa_pairs_chapter_7.json"

print("PDF path:", PDF_PATH)
print("Output directory:", OUTPUT_DIR)

PDF path: assets/Chapter_7.pdf
Output directory: artefacts


## Extracting Text from the PDF

The chapter content is first extracted from the PDF file so that it can be processed as plain text. This is necessary because retrieval systems operate on textual data rather than directly on PDF files. The extraction function opens the PDF using PyMuPDF and reads the text from each page sequentially. All page contents are then combined into one full document string. The raw extracted text is saved into a text file so that the original output can be inspected later if any preprocessing issues occur.

In [3]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """
    Extract text from all pages of a PDF file.

    Parameters:
        pdf_path (Path): Path to the PDF file.

    Returns:
        str: Combined text from all pages.
    """
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    all_pages = []

    # Open the PDF document
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text")
            all_pages.append(page_text)

    return "\n".join(all_pages)


# Extract raw text from the chapter PDF
raw_text = extract_text_from_pdf(PDF_PATH)

# Save raw text for inspection and reproducibility
RAW_TEXT_PATH.write_text(raw_text, encoding="utf-8")

print("Raw text length:", len(raw_text))
print("First 1000 characters of raw text:\n")
print(raw_text[:1000])

Raw text length: 91498
First 1000 characters of raw text:

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER
7
Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. Perhaps
this shouldn’t be surprising. Language is the mark
of humanity and sentience. conversation is the most
fundamental arena of language, the ﬁrst kind of lan-
guage we learn as children, and the kind we engage in
constantly, whether we

## Cleaning the Extracted Text

PDF-extracted text often contains noise such as inconsistent line breaks, unnecessary spaces, repeated blank lines, or isolated page numbers. These issues can negatively affect chunking and later retrieval quality. Therefore, the extracted text is cleaned before being used as the knowledge base. In this notebook, the cleaning process normalizes line breaks, removes lines that contain only page numbers, reduces repeated spaces, and compresses excessive blank lines. The goal is not to heavily alter the document, but to make it more consistent and suitable for RAG preprocessing while preserving the original meaning of the chapter content.

In [4]:
def clean_document_text(text: str) -> str:
    """
    Clean extracted PDF text for downstream RAG processing.

    Cleaning steps:
    1. Normalize line breaks
    2. Remove excessive spaces
    3. Remove repeated blank lines
    4. Remove obvious page number-only lines
    5. Strip leading/trailing whitespace
    """
    # Normalize different line break styles
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove lines that contain only page numbers
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Replace multiple spaces/tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove extra spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()


# Clean the raw text
clean_text = clean_document_text(raw_text)

# Save cleaned text
CLEAN_TEXT_PATH.write_text(clean_text, encoding="utf-8")

print("Cleaned text length:", len(clean_text))
print("First 1000 characters of cleaned text:\n")
print(clean_text[:1000])

Cleaned text length: 91402
First 1000 characters of cleaned text:

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER

Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. Perhaps
this shouldn’t be surprising. Language is the mark
of humanity and sentience. conversation is the most
fundamental arena of language, the ﬁrst kind of lan-
guage we learn as children, and the kind we engage in
constantly, whe

## Splitting the Document into Chunks

After cleaning, the chapter text is divided into smaller overlapping chunks. Chunking is a necessary preparation step for RAG because retrieval is performed on smaller text units rather than on the full document at once. In this notebook, a simple character-based chunking strategy is used with overlap between consecutive chunks. The overlap is important because it helps preserve context when a concept spans across chunk boundaries. Each chunk is stored together with a unique `chunk_id`, which will later make it easier to trace retrieved sources during answer generation.

In [5]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[dict]:
    """
    Split text into overlapping character-based chunks.

    Parameters:
        text (str): Full cleaned document text.
        chunk_size (int): Maximum size of each chunk in characters.
        overlap (int): Number of overlapping characters between chunks.

    Returns:
        list[dict]: List of chunk dictionaries with chunk_id and text.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append({
                "chunk_id": chunk_id,
                "text": chunk
            })
            chunk_id += 1

        # Move forward while keeping overlap
        start += chunk_size - overlap

    return chunks


# Create chunks from cleaned text
chunks = chunk_text(clean_text, chunk_size=1200, overlap=200)

# Save chunks to JSON
CHUNKS_PATH.write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

print("Number of chunks:", len(chunks))
print("\nSample chunk:\n")
pprint(chunks[0])

Number of chunks: 92

Sample chunk:

{'chunk_id': 0,
 'text': 'Speech and Language Processing.\n'
         'Daniel Jurafsky & James H. Martin.\n'
         'Copyright © 2026.\n'
         'All\n'
         'rights reserved.\n'
         'Draft of January 6, 2026.\n'
         'CHAPTER\n'
         '\n'
         'Large Language Models\n'
         '“How much do we know at any time? Much more, or so I believe, than '
         'we\n'
         'know we know.”\n'
         'Agatha Christie, The Moving Finger\n'
         'The literature of the fantastic abounds in inanimate objects '
         'magically endowed with\n'
         'the gift of speech. From Ovid’s statue of Pygmalion to Mary '
         'Shelley’s story about\n'
         'Frankenstein, we continually reinvent stories about\n'
         'creating something and then having a chat with it.\n'
         'Legend has it that after ﬁnishing his sculpture Moses,\n'
         'Michelangelo thought it so lifelike that he tapped it\n'
         'on the

## Inspecting Sample Chunks

Before moving to question-answer generation, it is useful to manually inspect a few chunks. This helps verify that the text cleaning and chunking process has worked properly. At this stage, the chunks should look readable, preserve the intended meaning of the original text, and avoid obvious extraction errors such as broken formatting or meaningless fragments. This manual inspection is also useful for debugging, since poor chunk quality can later reduce retrieval performance.

In [6]:
# Preview a few chunks to inspect quality
for i in range(min(3, len(chunks))):
    print(f"\n--- Chunk {i} ---\n")
    print(chunks[i]["text"][:700])


--- Chunk 0 ---

Speech and Language Processing.
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER

Large Language Models
“How much do we know at any time? Much more, or so I believe, than we
know we know.”
Agatha Christie, The Moving Finger
The literature of the fantastic abounds in inanimate objects magically endowed with
the gift of speech. From Ovid’s statue of Pygmalion to Mary Shelley’s story about
Frankenstein, we continually reinvent stories about
creating something and then having a chat with it.
Legend has it that after ﬁnishing his sculpture Moses,
Michelangelo thought it so lifelike that he tapped it
on the knee and commanded it to speak. 

--- Chunk 1 ---

our families or friends.
This chapter introduces the Large Language
Model, or LLM, a computational agent that can in-
teract conversationally with people. The fact that LLMs are designed for interaction
with people has strong implications for their design and use

## Preparing Question-Answer Pairs

The assignment requires at least 20 question-answer pairs based strictly on the assigned chapter. These QA pairs serve two purposes. First, the questions act as input queries to the RAG pipelines. Second, the answers act as ground-truth references for evaluation using ROUGE metrics in Task 2. For this reason, the QA pairs must be carefully written so that they are fully supported by the chapter content and not based on outside knowledge. A simple JSON structure is used to store the questions and their corresponding ground-truth answers.

In [7]:
# Template structure for QA pairs
qa_pairs = [
    # 5 Definition Questions
    {
        "question": "What is a Large Language Model (LLM)?",
        "ground_truth_answer": "An LLM is a computational agent or neural network that is designed to interact conversationally with people by predicting the next word from previous words in a given context or prefix. [cite: 13, 60, 78]"
    },
    {
        "question": "What is the definition of 'pretraining' in the context of LLMs?",
        "ground_truth_answer": "Pretraining is the process of learning knowledge about language and the world by iteratively predicting tokens in vast amounts of text. [cite: 41]"
    },
    {
        "question": "What is 'teacher forcing'?",
        "ground_truth_answer": "Teacher forcing is a training approach where the model is always given the correct history sequence to predict the next word, rather than feeding the model its own best guess from the previous time step. [cite: 562]"
    },
    {
        "question": "What is a 'prompt'?",
        "ground_truth_answer": "A prompt is a text string issued by a user to a language model to get the model to do something useful, such as answering a question or following an instruction. [cite: 219]"
    },
    {
        "question": "What is 'perplexity' in language model evaluation?",
        "ground_truth_answer": "Perplexity is a length-normalized metric used to evaluate how well a model predicts unseen text; it is the inverse probability a model assigns to a test set, normalized by the test set length. [cite: 701, 702]"
    },
    # 5 Explanation Questions
    {
        "question": "How does temperature sampling work?",
        "ground_truth_answer": "It reshapes the probability distribution by dividing the logits by a temperature parameter (τ) before the softmax; a low τ increases the probability of high-probability tokens, making the model more 'greedy.' [cite: 403, 405, 459]"
    },
    {
        "question": "How do children achieve high rates of vocabulary growth according to the text?",
        "ground_truth_answer": "Research suggests the bulk of vocabulary acquisition happens as a by-product of reading, which is a process of rich contextual processing rather than learning words in isolation. [cite: 32, 33]"
    },
    {
        "question": "How is an LLM turned from a predictive model into a generative one?",
        "ground_truth_answer": "It is turned into a generative model by repeatedly sampling from its output probability distribution and adding each generated token back into the context as a prefix for the next prediction. [cite: 101, 103]"
    },
    {
        "question": "How does 'in-context learning' differ from standard training?",
        "ground_truth_answer": "In-context learning improves performance through the provided context and activations in the network without involving gradient-based updates to the model's underlying parameters. [cite: 263, 265]"
    },
    {
        "question": "How is 'alignment' performed in the three-stage training process?",
        "ground_truth_answer": "The model is trained on preference data (labeled 'accepted' vs. 'rejected' continuations) using reinforcement learning or reward-based algorithms to make it maximally helpful and less harmful. [cite: 514, 516]"
    },
    # 4 Comparison Questions
    {
        "question": "What is the difference between an Encoder and a Decoder architecture?",
        "ground_truth_answer": "Decoders generate novel output tokens one at a time from left-to-right (generative), whereas Encoders produce vector representations for tokens and are typically used for classification rather than generation. [cite: 135, 136, 146, 151]"
    },
    {
        "question": "How does greedy decoding compare to random sampling?",
        "ground_truth_answer": "Greedy decoding always chooses the single most likely token and is deterministic, while random sampling chooses tokens according to their probability distribution, introducing more diversity. [cite: 321, 352, 359]"
    },
    {
        "question": "What is the difference between zero-shot and few-shot prompting?",
        "ground_truth_answer": "Zero-shot prompting provides instructions without labeled examples, while few-shot prompting includes labeled examples (demonstrations) within the prompt to help the model perform the task. [cite: 234, 235]"
    },
    {
        "question": "How do modern LLMs differ from traditional n-gram models?",
        "ground_truth_answer": "N-gram models predict words based only on a handful of previous words (like bigrams or trigrams), whereas LLMs can use contexts of thousands or tens of thousands of words. [cite: 80, 81]"
    },
    # 3 Importance/Use-case Questions
    {
        "question": "Why is the 'system prompt' important for LLM interaction?",
        "ground_truth_answer": "The system prompt is a first instruction that defines the task, role, tone, and overall context for the LM, and it is silently prepended to all user text. [cite: 266, 267]"
    },
    {
        "question": "What is the importance of the 'distributional hypothesis' for LLMs?",
        "ground_truth_answer": "It proposes that meaning can be learned from text based on the complex association of words with their co-occurring words, allowing models to acquire knowledge simply from reading. [cite: 36, 37]"
    },
    {
        "question": "Why is quality filtering necessary for pretraining corpora?",
        "ground_truth_answer": "Filtering removes boilerplate text, adult content, personal identifiable information (PII), and duplicate documents, which generally improves the performance of the language model. [cite: 634, 635, 637]"
    },
    # 3 Limitation/Challenge Questions
    {
        "question": "What is 'hallucination' in the context of LLMs?",
        "ground_truth_answer": "Hallucination is a safety issue where LLMs generate text that is false or incorrect because the training algorithm lacks a mechanism to enforce factuality. [cite: 778, 779]"
    },
    {
        "question": "What is the challenge of 'data contamination' in model evaluation?",
        "ground_truth_answer": "Data contamination occurs when test set data makes its way into the training set, leading the evaluation metrics to overstate the model's actual performance. [cite: 751, 753]"
    },
    {
        "question": "What are the primary ethical concerns regarding pretraining data scraped from the web?",
        "ground_truth_answer": "Key concerns include potential copyright violations, lack of data consent from website owners, privacy issues from leaked PII, and demographic skew. [cite: 647, 649, 652, 655]"
    }
]

# Save initial template
# Note: QA_PATH should be defined as a Path object in your environment
# QA_PATH.write_text(json.dumps(qa_pairs, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Generated 20 QA pairs based on the textbook content.")

Generated 20 QA pairs based on the textbook content.
